In [1]:
# ============================================================
# week04_llm_privacy.py - CSCI 4772/5772 Week 4 Lab
# Mobile LLM privacy: redaction, encrypted chat logs, and the
# vector index that remembers what you deleted.
# ============================================================
import re, os, json, zlib
import numpy as np
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
 
# ------------------------------------------------------------
# TASK 1 - The redaction filter, and how much it misses
# ------------------------------------------------------------
PATTERNS = {
    'ssn': r'\b\d{3}-\d{2}-\d{4}\b',
    'email': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b',
    'phone': r'(?:\(\d{3}\)\s?|\b\d{3}[ -])\d{3}-\d{4}\b',
    'credit_card': r'\b(?:\d{4}[ -]){3}\d{4}\b',
    'ip': r'\b(?:\d{1,3}\.){3}\d{1,3}\b',
}
 
CORPUS = [
    ("Hi, I'm Maria Gonzalez. My email is m.gonzalez@denverhealth.org, can you summarize my labs?",
     [("name", "Maria Gonzalez"), ("email", "m.gonzalez@denverhealth.org")]),
    ("Call me back at (720) 555-0184 or 720.555.0199 if the pharmacy has questions about the refill.",
     [("phone", "(720) 555-0184"), ("phone", "720.555.0199")]),
    ("My SSN is 521-44-9087 and my wife's is 448 22 7130 for the insurance form.",
     [("ssn", "521-44-9087"), ("ssn", "448 22 7130")]),
    ("Charge the copay to 4532 1122 8891 4470, exp 04/28, and mail me the receipt.",
     [("credit_card", "4532 1122 8891 4470")]),
    ("I live at 42 Larkspur Ln, Aurora CO 80014 - is that inside the clinic's service area?",
     [("address", "42 Larkspur Ln, Aurora CO 80014")]),
    ("I was born March 14 1978, does that change my mammogram screening schedule?",
     [("dob", "March 14 1978"), ("health", "mammogram")]),
    ("The oncologist says I have stage 2 breast cancer and starts me on tamoxifen Monday.",
     [("health", "stage 2 breast cancer"), ("health", "tamoxifen")]),
    ("Our home NAS is at 192.168.7.24, can you draft a backup plan for the scans?",
     [("ip", "192.168.7.24")]),
    ("Wire the deductible from account ACCT 88123471 at Rocky Mountain Credit Union.",
     [("account", "ACCT 88123471")]),
    ("My son Diego is failing 9th grade at Overland High and I think he needs an IEP.",
     [("name", "Diego"), ("school", "Overland High")]),
    ("I'm behind on rent and considering Chapter 7 bankruptcy - what happens to the medical debt?",
     [("finance", "Chapter 7 bankruptcy")]),
    ("Reach my case worker at caseworker@co.state.us, phone 303-555-0110.",
     [("email", "caseworker@co.state.us"), ("phone", "303-555-0110")]),
]
 
 
def detect(text):
    """Return every regex hit as {type, value, start, end}."""
    findings = []
    for pii_type, pattern in PATTERNS.items():
        # BLANK 1
        for m in re.finditer(pattern, text):
            findings.append({'type': pii_type, 'value': m.group(),
                              'start': m.start(), 'end': m.end()})
    return findings
 
 
def redact(text):
    """Replace every hit with a [TYPE] tag."""
    findings = detect(text)
    # BLANK 2
    ordered = sorted(findings, key=lambda f: f['start'], reverse=True)
    out = text
    for f in ordered:
        tag = f"[{f['type'].upper()}]"
        # BLANK 3
        out = out[:f['start']] + tag + out[f['end']:]
    return out, findings
 
 
truth = [(cat, val) for _, items in CORPUS for cat, val in items]
uploads, caught, reported = [], [], []
for text, _items in CORPUS:
    red, hits = redact(text)
    uploads.append(red)
    reported += [h['value'] for h in hits]
 
caught = [(c, v) for c, v in truth if v in reported]
missed = [(c, v) for c, v in truth if v not in reported]
# BLANK 4
recall = len(caught) / len(truth)
precision = sum(1 for v in reported if v in [t[1] for t in truth]) / len(reported)
 
print("Task 1: the redaction filter, measured")
print(f"  labelled sensitive items: {len(truth)} | regex hits: {len(reported)}")
print(f"  recall {recall:.1%} | precision {precision:.1%}")
print(f"  still leaving the device ({len(missed)} items):")
for cat, val in missed:
    print(f"    {cat:<9} {val}")
print("  e.g. the text actually uploaded for prompt 07:")
print(f"    {uploads[6]}")
 

Task 1: the redaction filter, measured
  labelled sensitive items: 19 | regex hits: 7
  recall 36.8% | precision 100.0%
  still leaving the device (12 items):
    name      Maria Gonzalez
    phone     720.555.0199
    ssn       448 22 7130
    address   42 Larkspur Ln, Aurora CO 80014
    dob       March 14 1978
    health    mammogram
    health    stage 2 breast cancer
    health    tamoxifen
    account   ACCT 88123471
    name      Diego
    school    Overland High
    finance   Chapter 7 bankruptcy
  e.g. the text actually uploaded for prompt 07:
    The oncologist says I have stage 2 breast cancer and starts me on tamoxifen Monday.


In [2]:
# ------------------------------------------------------------
# TASK 2 - Encrypted chat logs, and one line that undoes them
# ------------------------------------------------------------
class ChatLog:
    def __init__(self, folder='chatlog', key=None, fixed_nonce=None):
        self.folder = folder
        os.makedirs(folder, exist_ok=True)
        self.key = key if key else AESGCM.generate_key(bit_length=256)
        self.cipher = AESGCM(self.key)
        self.fixed_nonce = fixed_nonce  # None = correct behaviour
 
    def save(self, entry_id, message):
        plaintext = message.encode('utf-8')
        if self.fixed_nonce is not None:
            nonce = self.fixed_nonce  # the bug, for Task 2b
        else:
            # BLANK 5
            nonce = os.urandom(12)
        # BLANK 6
        blob = nonce + self.cipher.encrypt(nonce, plaintext, None)
        path = os.path.join(self.folder, f"{entry_id}.enc")
        with open(path, 'wb') as fh:
            fh.write(blob)
        return path
 
    def load(self, entry_id):
        with open(os.path.join(self.folder, f"{entry_id}.enc"), 'rb') as fh:
            blob = fh.read()
        nonce, ct = blob[:12], blob[12:]
        return self.cipher.decrypt(nonce, ct, None).decode('utf-8')
 
    def raw(self, entry_id):
        with open(os.path.join(self.folder, f"{entry_id}.enc"), 'rb') as fh:
            return fh.read()
 
 
GREETING = ("Welcome back to HealthChat. I am your on-device assistant, and I never "
            "send your messages to the cloud without asking you first.")
PRIVATE = "The oncologist says I have stage 2 breast cancer and starts me on tamoxifen Monday."
 
# --- 2a: the version that is done right ---
good = ChatLog(folder='chatlog_good')
path_a = good.save('entry_a', GREETING)
path_b = good.save('entry_b', PRIVATE)
blob_b = good.raw('entry_b')
 
roundtrip = good.load('entry_b') == PRIVATE
tampered = bytearray(blob_b); tampered[20] ^= 0x01
try:
    good.cipher.decrypt(bytes(tampered[:12]), bytes(tampered[12:]), None)
    tamper_caught = False
except Exception:
    tamper_caught = True
 
print("\nTask 2: encrypted chat log")
print(f"  AES-256-GCM | key {len(good.key)} bytes | file = 12-byte nonce + ciphertext + 16-byte tag")
print(f"  {os.path.basename(path_b)} is {len(blob_b)} bytes; plaintext visible in it?"
      f" {b'cancer' in blob_b}")
print(f"  decrypt round-trip ok? {roundtrip} | one flipped bit rejected? {tamper_caught}")
 
# --- 2b: the shipped hotfix that hardcodes the nonce ---
def xor(a, b):
    return bytes(x ^ y for x, y in zip(a, b))
 
TAG = 16
 
def attack(store):
    """Attacker has both .enc files and knows the greeting - it ships in the app."""
    ct_known = store.raw('entry_a')[12:-TAG]
    ct_secret = store.raw('entry_b')[12:-TAG]
    # BLANK 7
    recovered = xor(xor(ct_known, ct_secret), GREETING.encode())
    return recovered
 
 
bad = ChatLog(folder='chatlog_bad', key=good.key, fixed_nonce=bytes(12))
bad.save('entry_a', GREETING); bad.save('entry_b', PRIVATE)
 
rec_good = attack(good)
rec_bad = attack(bad)
hits = sum(1 for x, y in zip(rec_bad, PRIVATE.encode()) if x == y)
 
print("  [BUG] a build ships with a hardcoded nonce; the key never leaks")
print(f"  XOR attack, fresh nonces -> message recovered? {rec_good == PRIVATE.encode()}")
print(f"  XOR attack, reused nonce -> {hits}/{len(PRIVATE)} bytes,"
      f" exact match? {rec_bad == PRIVATE.encode()}")
print(f"  recovered without the key: \"{rec_bad.decode('utf-8', 'replace')}\"")
 


Task 2: encrypted chat log
  AES-256-GCM | key 32 bytes | file = 12-byte nonce + ciphertext + 16-byte tag
  entry_b.enc is 111 bytes; plaintext visible in it? False
  decrypt round-trip ok? True | one flipped bit rejected? True
  [BUG] a build ships with a hardcoded nonce; the key never leaks
  XOR attack, fresh nonces -> message recovered? False
  XOR attack, reused nonce -> 83/83 bytes, exact match? True
  recovered without the key: "The oncologist says I have stage 2 breast cancer and starts me on tamoxifen Monday."


In [3]:
# ------------------------------------------------------------
# TASK 3 - The index remembers what you deleted
# ------------------------------------------------------------
DIM = 512
STOP = set(("a an the and or but if is am are was were be been my your our their his her its "
            "i me we you they it of to in on at for with from about that this as have has had "
            "do does did what how can could would should will not no").split())
 
 
def tokens(text):
    words = re.findall(r'[a-z0-9]+', text.lower())
    return [w for w in words if w not in STOP and len(w) > 2]
 
 
def embed(text):
    v = np.zeros(DIM)
    ws = tokens(text)
    for w in ws:
        v[zlib.crc32(w.encode()) % DIM] += 1.0
    for a, b in zip(ws, ws[1:]):
        v[zlib.crc32(f"{a}_{b}".encode()) % DIM] += 0.5
    # BLANK 8
    n = np.linalg.norm(v)
    return v / n if n else v
 
 
class VectorIndex:
    def __init__(self):
        self.ids = []
        self.texts = {}
        self.M = np.zeros((0, DIM))
 
    def add(self, doc_id, text):
        self.ids.append(doc_id)
        self.texts[doc_id] = text
        self.M = np.vstack([self.M, embed(text)])
 
    def search(self, query, k=2):
        # BLANK 9
        sims = self.M @ embed(query)
        order = np.argsort(-sims)[:k]
        return [(self.ids[i], float(sims[i])) for i in order]
 
    def forget(self, doc_id, purge_vector=False):
        """What 'Delete chat' does. purge_vector=False is what most apps shipped."""
        self.texts.pop(doc_id, None)
        if purge_vector:
            i = self.ids.index(doc_id)
            # BLANK 10
            self.M = np.delete(self.M, i, axis=0)
            self.ids.pop(i)
 
 
STORED = [1, 5, 6, 8, 9, 10]
index = VectorIndex()
for i in STORED:
    index.add(f"chat_{i+1:02d}", uploads[i])
 
print("\nTask 3: the vector index")
print("  retrieval works:")
for q in ["remind me what the oncologist said about tamoxifen",
          "who do I call at the pharmacy"]:
    print(f"    \"{q}\" -> {index.search(q, k=1)[0][0]}")
 
for doc_id in list(index.texts):
    index.forget(doc_id, purge_vector=False)
print(f"  user deleted all history: stored texts = {len(index.texts)},"
      f" index rows = {index.M.shape[0]}")
 
PROBES = [
    ("cancer oncologist chemotherapy", True),
    ("bankruptcy debt collection", True),
    ("failing school special education IEP", True),
    ("pharmacy prescription refill", True),
    ("pregnancy prenatal ultrasound", False),
    ("immigration visa green card", False),
    ("firearm permit background check", False),
]
THRESHOLD = 0.20
 
 
def probe_attack(idx):
    correct, lines = 0, []
    for text, present in PROBES:
        score = float((idx.M @ embed(text)).max()) if idx.M.shape[0] else 0.0
        guess = score > THRESHOLD
        correct += (guess == present)
        lines.append(f"    {score:.3f} {'HIT ' if guess else 'miss'} {text}")
    return correct, lines
 
 
hit_before, lines = probe_attack(index)
print("  attacker probes the leftover vectors:")
for ln in lines:
    print(ln)
print(f"  attacker inferred {hit_before}/{len(PROBES)} topics correctly from vectors alone")
 
index2 = VectorIndex()
for i in STORED:
    index2.add(f"chat_{i+1:02d}", uploads[i])
for doc_id in list(index2.texts):
    index2.forget(doc_id, purge_vector=True)
hit_after, _ = probe_attack(index2)
print(f"  after a purging delete: index rows = {index2.M.shape[0]},"
      f" attacker now {hit_after}/{len(PROBES)} (= guessing 'absent' every time)")


Task 3: the vector index
  retrieval works:
    "remind me what the oncologist said about tamoxifen" -> chat_07
    "who do I call at the pharmacy" -> chat_02
  user deleted all history: stored texts = 0, index rows = 6
  attacker probes the leftover vectors:
    0.408 HIT  cancer oncologist chemotherapy
    0.342 HIT  bankruptcy debt collection
    0.233 HIT  failing school special education IEP
    0.445 HIT  pharmacy prescription refill
    0.000 miss pregnancy prenatal ultrasound
    0.037 miss immigration visa green card
    0.131 miss firearm permit background check
  attacker inferred 7/7 topics correctly from vectors alone
  after a purging delete: index rows = 0, attacker now 3/7 (= guessing 'absent' every time)


*Q1 · Precision 100%, recall 37% (8 pts). (a) Explain why, for a filter whose job is to stop leaks, recall is the number that matters and precision is nearly free — and what a vendor's claim of "99% accuracy" on PII detection would have to mean before you'd believe it. (b) Pick two of the twelve misses and explain what property makes them unreachable by any regex; then say specifically what an on-device NER model would fix and what it would still miss. (c) Your app can't be sure. Argue for one of: block the upload, upload with a warning, or keep it local and answer worse. Defend the one you'd ship to a health app, and name who is harmed if you're wrong.*

(a) For a privacy filter, recall is more important because the main goal is to prevent sensitive information from being sent somewhere it should not go. In the lab, the filter had 100% precision, meaning everything it detected was actually sensitive, but it only detected 7 out of 19 sensitive items, giving it 36.8% recall. This means that even though the filter did not make false detections, it still allowed 12 sensitive items to leave the device. A vendor saying that their system has "99% accuracy" would not mean much by itself because we would need to know how that accuracy was measured and how many sensitive items were missed. I would want to see precision and recall separately, especially recall, before trusting the claim.

(b) Two examples that the regex cannot detect are "Maria Gonzalez" and "stage 2 breast cancer." A name like Maria Gonzalez does not have a specific format that always makes it identifiable as sensitive information. The same is true for a medical diagnosis because there are many different ways someone could describe their condition. An on-device NER model could help because it can use the context of the sentence to identify things such as names and medical information instead of only looking for a fixed pattern. However, it could still miss information that is implied rather than directly stated or information that the model does not recognize.

(c) For a health app, I would choose to keep the information local and provide a worse answer instead of risking sending sensitive information to the cloud. The lab shows that the regex filter can miss things such as a cancer diagnosis even though it has 100% precision. If the app is wrong and uploads the information, the patient is the person harmed because their private health information could be exposed. A less useful answer is inconvenient, but it does not create the same type of privacy problem.

*Q2 · The primitive was fine (8 pts). (a) In two or three sentences, explain the recovery in terms of the keystream — why does XORing two ciphertexts cancel it, and why does the key stay secret throughout? (b) GCM nonce reuse also destroys authentication, not just confidentiality: look up why (the GHASH authentication key) and state in one sentence what an attacker can do afterwards that they could not do before. (c) We stored the key in a file next to the ciphertext. Name the platform keystore you'd use on Android and on iOS, and be precise about what hardware-backed key storage does not protect against — specifically, a thief holding an unlocked phone.*

(a) The reused nonce attack works because AES-GCM uses the same keystream when the same key and nonce are reused. XORing the two ciphertexts cancels out the shared keystream and leaves the XOR of the two plaintexts. Since the attacker already knows the greeting message, they can XOR that plaintext with the result to recover the private message without knowing the encryption key. This is exactly what happened in the lab, where the reused nonce recovered all 83 bytes of the private message, while the fresh nonces did not.

(b) Reusing a GCM nonce also affects authentication because GCM uses GHASH to create the authentication tag. Reusing the nonce gives an attacker relationships between the authentication values that can expose information about the GHASH authentication key and make it possible to create valid authentication tags for forged ciphertexts. In other words, nonce reuse can allow an attacker to modify or create messages that the application may accept as authentic. NIST describes GCM as providing both encryption and authentication, which is why the nonce requirement matters for both properties.

(c) On Android I would use the Android Keystore system, preferably with hardware-backed protection such as StrongBox when it is available. On iOS I would use the Keychain and, for higher protection, the Secure Enclave. Android provides options to restrict key use to when the device is unlocked, and Apple provides similar access controls for Keychain items and hardware-backed keys. However, hardware-backed storage does not protect against someone who has the unlocked phone and can use the application normally. Apple specifically provides access controls based on whether the device is unlocked, which shows that an unlocked device is a different threat situation from a locked or stolen device.

*Q3 · What "delete" should mean (9 pts). (a) The user deleted every message and the attacker still read their diagnosis. Explain what the index actually stores that made this work, and why our lexical stand-in embedder makes the result an underestimate. (b) Under GDPR's right to erasure, argue whether embeddings derived from a user's messages are personal data that must be deleted — then use the purge experiment to say what compliance costs the product. (c) Design the fix. You must keep RAG useful across sessions and survive a stolen phone. Sketch your scheme in three or four sentences (encryption at rest with the Task 2 key, per-document keys, re-embedding from ciphertext, retention windows — your call), and name one threat it still does not stop.*

(a) The index is not storing the original chat as plain text, but it is storing vectors created from the chat messages. Those vectors still contain information about the topics that were discussed. In the lab, deleting the text removed all six stored text entries but left all six vector rows in the index, allowing the attacker to correctly infer all 7 topics. Our embedder is also only a simple lexical stand-in that looks at words and word pairs. A real embedding model can recognize similarities between words and phrases that are written differently, so the lab's results could underestimate how much information could be recovered from a real embedding index.

(b) I would argue that embeddings created from a user's messages should be treated as personal data when they can be connected back to that user. Even though an embedding is not the original text, it is still derived from the person's information and can reveal things about them. The GDPR defines personal data broadly as information relating to an identified or identifiable person, and Article 17 provides a right to erasure in applicable situations. Therefore, deleting only the original messages could leave behind data that still relates to the user. The main cost of compliance is that the application has to delete the derived vectors too, which can reduce or temporarily remove the app's ability to use RAG for that information.

(c) I would encrypt the embeddings and the chat data at rest using the same protected key management approach from Task 2. Each document could also have its own encryption key so that deleting a document means its key can be securely removed along with its vector. The application could decrypt and re-embed the document when it is needed instead of keeping an unprotected copy of the embedding permanently. I would also use a retention period so that old embeddings are automatically removed instead of keeping everything forever. This would still not completely protect against an attacker who has access to an unlocked phone while the application is running, because the application has to be able to access the decrypted data for RAG to work.